In [1]:
#Loadin the forecast dataset
import pandas as pd

model_data = pd.read_csv(
    "../data/processed/cement_forecasting_model_data.csv",
    parse_dates=["date"]
)

model_data.shape

(32040, 22)

In [2]:
last_training_date = model_data["date"].max() - pd.Timedelta(days=56)

train_data = model_data[model_data["date"] <= last_training_date]
test_data = model_data[model_data["date"] > last_training_date]

train_data.shape, test_data.shape

((30360, 22), (1680, 22))

In [3]:
#Selecting Model inputs
model_features = [
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "cement_type",
    "region",
    "behavior",
    "year",
    "month",
    "week_of_year",
    "day_of_week"
]

In [6]:
X_train = pd.get_dummies(train_data[model_features]) # converting the categorical variables into numerical
X_test = pd.get_dummies(test_data[model_features])

y_train = train_data["consumed_tonnes"]
y_test = test_data["consumed_tonnes"]

In [5]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((30360, 18), (1680, 18), (30360,), (1680,))

#### Random Forecast Model

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

# Model initialization
rf_model = RandomForestRegressor(random_state=2026)

# Model training
rf_model.fit(X_train, y_train)

# Model testing
rf_predictions = rf_model.predict(X_test)

# Model evaluation
non_zero = y_test > 0

rf_mape = mean_absolute_percentage_error(
    y_test[non_zero],
    rf_predictions[non_zero]
) * 100

rf_rmse = mean_squared_error(
    y_test,
    rf_predictions
) ** 0.5

# Print results
print("Random Forest MAPE:", round(rf_mape, 2), "%")
print("Random Forest RMSE:", round(rf_rmse, 2), "tonnes")

Random Forest MAPE: 19.76 %
Random Forest RMSE: 7.65 tonnes


In [8]:
#Saving the outcome of the random forecast for comparisom
rf_forecasts = test_data[
    ["date", "site_id", "consumed_tonnes"]
].copy()

rf_forecasts["predicted_tonnes"] = rf_predictions

rf_forecasts.to_csv(
    "../outputs/random_forest_forecasts.csv",
    index=False
)

### XGBoost

In [12]:
from xgboost import XGBRegressor

# Model initialization
xgb_model = XGBRegressor(random_state=2026)

# Model training
xgb_model.fit(X_train, y_train)

# Model testing
xgb_predictions = xgb_model.predict(X_test)

# Model evaluation
non_zero = y_test > 0

xgb_mape = mean_absolute_percentage_error(
    y_test[non_zero],
    xgb_predictions[non_zero]
) * 100

xgb_rmse = mean_squared_error(
    y_test,
    xgb_predictions
) ** 0.5

# Print results
print("XGBoost MAPE:", round(xgb_mape, 2), "%")
print("XGBoost RMSE:", round(xgb_rmse, 2), "tonnes")

XGBoost MAPE: 22.06 %
XGBoost RMSE: 8.12 tonnes


In [13]:
# Save the XGboost forecasts
xgb_forecasts = test_data[
    ["date", "site_id", "consumed_tonnes"]
].copy()

xgb_forecasts["predicted_tonnes"] = xgb_predictions

xgb_forecasts.to_csv(
    "../outputs/xgboost_forecasts.csv",
    index=False
)

### Prophet

In [17]:
from prophet import Prophet

In [18]:
all_prophet_forecasts = []

for current_site in train_data["site_id"].unique():

    current_train = train_data[
        train_data["site_id"] == current_site
    ]

    current_test = test_data[
        test_data["site_id"] == current_site
    ]

    prophet_train = current_train[
        [
            "date",
            "consumed_tonnes",
            "planned_pour_tonnes",
            "rain_mm",
            "avg_temp_c"
        ]
    ].rename(columns={
        "date": "ds",
        "consumed_tonnes": "y"
    })

    prophet_future = current_test[
        [
            "date",
            "planned_pour_tonnes",
            "rain_mm",
            "avg_temp_c"
        ]
    ].rename(columns={"date": "ds"})

    # Model initialization
    prophet_model = Prophet()

    prophet_model.add_regressor("planned_pour_tonnes")
    prophet_model.add_regressor("rain_mm")
    prophet_model.add_regressor("avg_temp_c")

    # Model training
    prophet_model.fit(prophet_train)

    # Model testing
    prophet_predictions = prophet_model.predict(prophet_future)

    # Store this site's results
    site_forecast = current_test[
        ["date", "site_id", "consumed_tonnes"]
    ].copy()

    site_forecast["predicted_tonnes"] = (
        prophet_predictions["yhat"].to_numpy()
    )

    all_prophet_forecasts.append(site_forecast)

23:47:49 - cmdstanpy - INFO - Chain [1] start processing
23:47:50 - cmdstanpy - INFO - Chain [1] done processing
23:47:50 - cmdstanpy - INFO - Chain [1] start processing
23:47:50 - cmdstanpy - INFO - Chain [1] done processing
23:47:50 - cmdstanpy - INFO - Chain [1] start processing
23:47:50 - cmdstanpy - INFO - Chain [1] done processing
23:47:51 - cmdstanpy - INFO - Chain [1] start processing
23:47:51 - cmdstanpy - INFO - Chain [1] done processing
23:47:51 - cmdstanpy - INFO - Chain [1] start processing
23:47:51 - cmdstanpy - INFO - Chain [1] done processing
23:47:51 - cmdstanpy - INFO - Chain [1] start processing
23:47:51 - cmdstanpy - INFO - Chain [1] done processing
23:47:52 - cmdstanpy - INFO - Chain [1] start processing
23:47:52 - cmdstanpy - INFO - Chain [1] done processing
23:47:52 - cmdstanpy - INFO - Chain [1] start processing
23:47:52 - cmdstanpy - INFO - Chain [1] done processing
23:47:52 - cmdstanpy - INFO - Chain [1] start processing
23:47:52 - cmdstanpy - INFO - Chain [1]

In [19]:
prophet_forecasts = pd.concat(
    all_prophet_forecasts,
    ignore_index=True
)

actual = prophet_forecasts["consumed_tonnes"]
predicted = prophet_forecasts["predicted_tonnes"]

non_zero = actual > 0

prophet_mape = mean_absolute_percentage_error(
    actual[non_zero],
    predicted[non_zero]
) * 100

prophet_rmse = mean_squared_error(
    actual,
    predicted
) ** 0.5

print("Prophet MAPE:", round(prophet_mape, 2), "%")
print("Prophet RMSE:", round(prophet_rmse, 2), "tonnes")

Prophet MAPE: 23.55 %
Prophet RMSE: 9.32 tonnes


In [20]:
#Saving the forecast result for Prophet
prophet_forecasts.to_csv(
    "../outputs/prophet_forecasts.csv",
    index=False
)